# 03 — Sổ Tay Đánh Giá Prompt `AiAdvisor`, Benchmark Giảm OpenAI & Xuất Bản BigQuery Gold AI Mart

**Thiết kế sẵn sàng cho Papermill (`parameters` cell) & chiến lược 3 năm của CTO:**
- Khi `OPENAI_API_KEY` là **biến tạm** (`sk-temp-placeholder-replace-me`), `create_ai_provider()` tự dùng `RuleBasedFallbackProvider` (`0đ` token) để chạy & kiểm định toàn bộ luồng.
- Khi điền `OPENAI_API_KEY` thật vào `.env`, hệ thống tự chuyển sang `OpenAiAdvisorProvider` (Structured Outputs khóa cứng Pydantic).
- Đầu ra luôn được kiểm định `0%` biến kỹ thuật `snake_case`, `0%` SQL, `0%` placeholder `[X]%` trước khi ghi vào bảng BigQuery Gold AI Mart (`mart_property_ai_insights`) cho `bigquery-bff` (`:4000`) & `Partner Dashboard` (`:5182`) đọc siêu tốc.

In [ ]:
# Cell gắn tag `parameters` cho Papermill chạy tự động T-1
RUN_DATE = "2026-11-06"
DATA_SOURCE_MODE = "auto"       # auto | bigquery | seed
AI_PROVIDER_MODE = "auto"       # auto | fallback | openai
ALLOW_BQ_WRITEBACK = False      # Để False để chạy Dry-Run an toàn

In [ ]:
import os
import sys
from dataclasses import replace
from pathlib import Path

cwd = Path.cwd().resolve()
NOTEBOOK_ROOT = next(
    (p for p in (cwd, cwd.parent, cwd / "analytics-notebooks") if (p / "src").is_dir()),
    cwd,
)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import pandas as pd

from src.ai_provider import create_ai_provider, validate_business_prose
from src.bq_client import GoodNstayAnalyticsLoader
from src.config import load_settings

base_settings = load_settings()
settings = replace(
    base_settings,
    data_source_mode=DATA_SOURCE_MODE,
    ai_provider_mode=AI_PROVIDER_MODE,
    allow_bq_writeback=ALLOW_BQ_WRITEBACK,
)
loader = GoodNstayAnalyticsLoader(settings)
provider = create_ai_provider(settings)

df_snaps, snap_meta = loader.load_ai_advisor_snapshots()
print("Snapshot Telemetry:", snap_meta)
print(f"AI Provider: {provider.provider_name} | Model: {provider.model_used} | Key: {settings.masked_openai_key()}")
df_snaps

## 1. Sinh 3 Thẻ Chẩn Đoán (*Tại sao Trúng vs. Không trúng*) & Kiểm Định Guardrail (`AGENTS.md` §8)

In [ ]:
eval_rows = []
publish_rows = []

for snap in df_snaps.to_dict(orient="records"):
    diagnosis = provider.diagnose_branch(snap)
    violations = validate_business_prose(diagnosis)
    assert len(violations) == 0, f"Guardrail violation for {diagnosis.branch_id}: {violations}"

    eval_rows.append(
        {
            "branch_id": diagnosis.branch_id,
            "primary_bottleneck": diagnosis.primary_bottleneck,
            "cards_count": len(diagnosis.cards),
            "guardrail_violations": len(violations),
            "provider": diagnosis.provider,
            "sample_headline_1": diagnosis.cards[0].headline_vi,
        }
    )
    publish_rows.append(
        {
            "metric_date": pd.to_datetime(RUN_DATE).date(),
            "partner_id": snap["partner_id"],
            "branch_id": diagnosis.branch_id,
            "primary_bottleneck": diagnosis.primary_bottleneck,
            "cards_json": diagnosis.model_dump_json(),
            "provider": diagnosis.provider,
            "model_used": diagnosis.model_used,
        }
    )

df_eval = pd.DataFrame(eval_rows)
df_insights = pd.DataFrame(publish_rows)
df_eval

## 2. Xuất Bản (Dry-Run / Controlled Writeback) Vào BigQuery Gold AI Mart

In [ ]:
publish_result = loader.publish_ai_insights(
    df_insights=df_insights,
    run_date=RUN_DATE,
    dry_run=not ALLOW_BQ_WRITEBACK,
)
publish_result